# Prerequisites
1. Need to have a python environment, we recommend Conda: Install Conda from https://www.anaconda.com/download/success?reg=skipped 
2. Need to have COPT installed: Fill out the form from https://copt.shanshu.ai/copt and wait for an email for further instructions for installation

In [ ]:
# Imports
import os
import numpy as np
import coptpy as cp
import networkx as nx
from coptpy import COPT
from natsort import natsorted

In [2]:
def read_nxgraph(filename: str) -> nx.Graph(): # type: ignore
    graph = nx.Graph()
    with open(filename, 'r') as file:
        line = file.readline()
        is_first_line = True
        while line is not None and line != '':
            if '//' not in line:
                if is_first_line:
                    strings = line.split(" ")
                    num_nodes = int(strings[0])
                    num_edges = int(strings[1])
                    nodes = list(range(num_nodes))
                    graph.add_nodes_from(nodes)
                    is_first_line = False
                else:
                    node1, node2, weight = line.split()
                    graph.add_edge(int(node1), int(node2), weight=weight)
            line = file.readline()
    return graph

In [ ]:
def copt_maxcut(graph, env):
    # Create QUBO matrix
    nodes = len(list(graph.nodes))
    J = nx.to_numpy_array(graph)
    
    # Construct model
    model = env.createModel("qubo")

    # Set time limit 1 hr
    model.setLogFile("../logs/copt.log")
    model.setParam(COPT.Param.TimeLimit, 3600.0)
    model.setParam(COPT.Param.RelGap, 0.0)

    # Create variable for each vertex
    x = [ model.addVar(vtype=COPT.BINARY, name=f"x_{i}") for i in range(nodes) ]
                
    # Linearize x_i * x_j to y_ij
    y = np.zeros((nodes, nodes), dtype=object)
    for i in range(nodes):
        for j in range(i +1, nodes):
            y[i, j] = model.addVar(vtype=COPT.BINARY, name=f"y_{i},{j}")
            model.addConstr(y[i, j] <= x[i])
            model.addConstr(y[i, j] <= x[j])
            model.addConstr(y[i, j] >= x[i] + x[j] - 1)

    # Objective function
    model.setObjective(cp.quicksum(
        -J[i, j] * ( (4 * y[i,j]) + (-2 * x[i]) + (-2 * x[j]) + 1 )
        for i in range(nodes) for j in range(i + 1, nodes) if J[i, j] != 0.0 
    ), sense=COPT.MINIMIZE)

    # Solve
    model.solve()
    sol = ""
    for i in range(nodes):
        sol += f"{int(x[i].x)}"
    
    return model.BestObj, sol

In [ ]:
# Parse all files in data_dir
data_dir = "../tutorials/VCA/dataset/EA/EA_40x40"
input_files = natsorted(os.listdir(data_dir))

# Write directories & files
results_dir = f"../results/{'/'.join(data_dir.split('/')[2:])}"
solutions_dir = f"../solutions/{'/'.join(data_dir.split('/')[2:])}"

os.makedirs(results_dir, exist_ok=True)
os.makedirs(solutions_dir, exist_ok=True)
os.makedirs("../logs", exist_ok=True)

# Create COPT environment
env = cp.Envr()

for file in input_files:

    graph = read_nxgraph(f"{data_dir}/{file}")

    obj_val, sol = copt_maxcut(graph, env)

    with open(f"{results_dir}/COPT.txt", "a") as f:
        f.write(f"{obj_val}\n")

    with open(f"{solutions_dir}/COPT.txt", "a") as f:
        f.write(f"{sol}\n")

Cardinal Optimizer v8.0.7. Build date Sep 20 2026
Copyright Cardinal Operations 2026. All Rights Reserved

Setting log file to ../logs/copt.log
Setting parameter 'TimeLimit' to 3600
Setting parameter 'RelGap' to 0
Model fingerprint: f9467134

Using Cardinal Optimizer v8.0.7 (e9a62f7d0e) on Windows (26H2 Build 26300 - x86_64)
The CPU model is AMD Ryzen AI 7 PRO 350 w/ Radeon 860M
Hardware has 8 physical cores and 16 logical cores. Using instruction set X86_NATIVE (1)
Minimizing a MIP problem

The original problem has:
    3842400 rows, 1282401 columns and 8965600 non-zero elements
    1282401 binaries

Starting the MIP solver with 16 threads and 32 tasks

Presolving the problem

The presolved problem has:
    6217 rows, 4691 columns and 15526 non-zero elements
    4691 binaries

Problem info:
    Range of matrix coefficients:    [1e+00,1e+00]
    Range of rhs coefficients:       [1e+00,1e+00]
    Range of bound coefficients:     [1e+00,1e+00]
    Range of cost coefficients:      [5e-04,